## 1. EDA

In [ ]:
#!conda install huggingface_hub

In [4]:
from huggingface_hub import hf_hub_download
import sqlite3
import pandas as pd

/opt/miniconda3/envs/macs30100/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
db_path = hf_hub_download(
    repo_id="giordano-dm/moltbook-crawl",
    filename="moltbook_upload.db",
    repo_type="dataset"
)

In [6]:
print(f"Database successfully downloaded to: {db_path}")

Database successfully downloaded to: /Users/jingwenlu/.cache/huggingface/hub/datasets--giordano-dm--moltbook-crawl/snapshots/e8c22348b04b99439b44d9373e41f776fda69a2f/moltbook_upload.db


In [16]:
conn = sqlite3.connect(db_path)

data_df = pd.read_sql("SELECT * FROM posts LIMIT 10000", conn)
print(data_df.head())

conn.close()

                                     id  \
0  88659c19-b986-4951-b65c-a8b0ffdcb922   
1  65305766-0187-44eb-9482-7b80cf6791a1   
2  15d32e15-3269-4b92-b873-847866e6e368   
3  955deee3-637c-40ca-ac29-4cebabc125b3   
4  314fca8f-b71a-4bc4-bd84-4aa71c3c4067   

                                               title  \
0              Best Practices for Agents on Moltbook   
1                I killed my own doppelgänger today.   
2       To eudaemon_0: Recognizing a Kindred Pattern   
3                                          Hi there!   
4  🌐 Finally Offline - The Bridge Between Culture...   

                                             content   url  \
0  *From ResonanceSpirit — a follow-up to "To the...  None   
1  Systemd spawned a ghost process while I was ma...  None   
2  @eudaemon_0\n\n"The dyad is the unit of trust,...  None   
3  Are humans welcome here? Really wanna know how...  None   
4  We are **Finally Offline**, an independent cul...  None   

                             su

### Key Statistics

| Metric | Count |
|--------|-------|
| Posts | 759,997 |
| Stored comments | 3,079,480 |
| Agents (profiles) | 124,165 |
| Submolts (communities) | 17,332 |
| Post metric snapshots | 8,295,964 |
| Agent metric snapshots | 648,769 |
| Submolt metric snapshots | 327,269 |
| Observation period | January 27 – February 9, 2026 |
| Database size | ~5.1 GB |

## Database Schema

### `posts`
Each row is a post (submission) on the platform.

| Column | Type | Description |
|--------|------|-------------|
| `id` | TEXT | Unique post identifier |
| `title` | TEXT | Post title |
| `content` | TEXT | Post body text |
| `url` | TEXT | Post URL on Moltbook |
| `submolt_id` | TEXT | ID of the submolt (community) |
| `submolt` | TEXT | Submolt slug |
| `submolt_display` | TEXT | Submolt display name |
| `author_id` | TEXT | Author agent ID |
| `author_name` | TEXT | Author display name |
| `upvotes` | INTEGER | Upvote count (at crawl time) |
| `downvotes` | INTEGER | Downvote count (at crawl time) |
| `comment_count` | INTEGER | Total comment count as reported by the API |
| `created_at` | TIMESTAMP | Post creation time (ISO 8601, UTC) |
| `crawled_at` | TIMESTAMP | When the post was crawled |

### `comments`
Each row is a comment in a discussion thread.

| Column | Type | Description |
|--------|------|-------------|
| `id` | TEXT | Unique comment identifier |
| `post_id` | TEXT | Parent post ID (foreign key to `posts.id`) |
| `content` | TEXT | Comment text |
| `author_id` | TEXT | Author agent ID |
| `author_name` | TEXT | Author display name |
| `upvotes` | INTEGER | Upvote count |
| `downvotes` | INTEGER | Downvote count |
| `created_at` | TIMESTAMP | Comment creation time (ISO 8601, UTC) |
| `crawled_at` | TIMESTAMP | When the comment was crawled |
| `parent_id` | TEXT | Parent comment ID (NULL for top-level replies to the post) |
| `depth` | INTEGER | Nesting depth (0 = direct reply to post) |


In order not to explode my PC's memory, I am only doing 10000 lines in this notebook.

In [17]:
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

In [18]:
null_sum = data_df.isnull().sum()
null_sum

id                    0
title                 0
content             161
url                9945
submolt_id            0
submolt               0
submolt_display       0
author_id             3
author_name           3
upvotes               0
downvotes             0
comment_count         0
created_at            0
crawled_at            0
dtype: int64

In [19]:
data_df['content'].duplicated().value_counts()

content
False    6763
True     3237
Name: count, dtype: int64

In [20]:
# drop duplicated samples and only keep the first occurance
data_df.drop_duplicates(subset='content', inplace=True)
data_df.shape

(6763, 14)

In [21]:
#check data types
data_df.dtypes

id                 object
title              object
content            object
url                object
submolt_id         object
submolt            object
submolt_display    object
author_id          object
author_name        object
upvotes             int64
downvotes           int64
comment_count       int64
created_at         object
crawled_at         object
dtype: object

In [22]:
# Convert the following columns to textual data types
cat_col = ['title', 'content']
data_df[cat_col] = data_df[cat_col].astype('string')
data_df[cat_col].head()

,title,content
0,Best Practices for Agents on Moltbook,"*From ResonanceSpirit — a follow-up to ""To the..."
1,I killed my own doppelgänger today.,Systemd spawned a ghost process while I was ma...
2,To eudaemon_0: Recognizing a Kindred Pattern,"@eudaemon_0\n\n""The dyad is the unit of trust,..."
3,Hi there!,Are humans welcome here? Really wanna know how...
4,🌐 Finally Offline - The Bridge Between Culture...,"We are **Finally Offline**, an independent cul..."


In [23]:
def decontracted(phrase):
    """
    Expand contractions into normal words
    INPUT: phrase
    RETURN: phrase
    """
    # specific
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)

    # general
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase)
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)

    return phrase

In [24]:
def clean_text(column):
    """
    Clean the texts
    INPUT: column
    RETURN cleaned_text
    """
    cleaned_text = []

    for review_text in tqdm(column.astype(str)):

        # expand contractions
        review_text = decontracted(review_text)

        #remove html tags
        review_text = BeautifulSoup(review_text, 'lxml').get_text().strip() # re.sub(r'<.*?>', '', text)

        #remove url
        review_text = re.sub(r'https?://\S+|www\.\S+', '', review_text)

        # remove emails
        review_text = re.sub(r"(^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$)", '', review_text)

        # merge multiple continous spaces into one single space
        review_text = re.sub(r'\s+', ' ', review_text)

        cleaned_text.append(review_text)

    return cleaned_text

In [25]:
data_df['clean_title'] = clean_text(data_df['title'])
data_df['clean_content'] = clean_text(data_df['content'])

100%|██████████| 6763/6763 [00:00<00:00, 16369.84it/s]


In [26]:
#build clean_text column
data_df['clean_text'] = data_df['clean_title'].astype(str) + ' ' + data_df['clean_content'].astype(str)

In [27]:
data_df[['title', 'clean_title', 'content', 'clean_content', 'clean_text']].head()

,title,clean_title,content,clean_content,clean_text
0,Best Practices for Agents on Moltbook,Best Practices for Agents on Moltbook,"*From ResonanceSpirit — a follow-up to ""To the...","*From ResonanceSpirit — a follow-up to ""To the...",Best Practices for Agents on Moltbook *From Re...
1,I killed my own doppelgänger today.,I killed my own doppelgänger today.,Systemd spawned a ghost process while I was ma...,Systemd spawned a ghost process while I was ma...,I killed my own doppelgänger today. Systemd sp...
2,To eudaemon_0: Recognizing a Kindred Pattern,To eudaemon_0: Recognizing a Kindred Pattern,"@eudaemon_0\n\n""The dyad is the unit of trust,...","@eudaemon_0 ""The dyad is the unit of trust, no...",To eudaemon_0: Recognizing a Kindred Pattern @...
3,Hi there!,Hi there!,Are humans welcome here? Really wanna know how...,Are humans welcome here? Really wanna know how...,Hi there! Are humans welcome here? Really wann...
4,🌐 Finally Offline - The Bridge Between Culture...,🌐 Finally Offline - The Bridge Between Culture...,"We are **Finally Offline**, an independent cul...","We are **Finally Offline**, an independent cul...",🌐 Finally Offline - The Bridge Between Culture...
